# SQL for Data Platforms
## Part 10: Simulating a Platform Locally (dbt & Governance)

Every module so far ran against SQLite in-memory. This module compares your local-simulation
options honestly, then drives a **real dbt-duckdb project** — seeds, staging/mart models, a macro,
generic + singular tests, an incremental model — showing dbt's tests/docs/DAG as a lightweight
governance layer you get for free on a laptop. See
[Governance, Catalog & Lineage](../../mfzamudio.github.io/publications/pattern-governance-lineage.html)
on the main site for the platform-scale version of the same idea (Unity Catalog, Purview, etc.).

## Section 1 — Choosing a local simulation option

| Option | What it is | When to reach for it |
|---|---|---|
| **SQLite in-memory** | Zero-install, stdlib `sqlite3`, `:memory:` | Fast iteration on plain relational SQL — what Parts 1–9 used |
| **DuckDB** | Zero-install, `pip install duckdb`, single-file or in-memory | Anything columnar/analytical: Parquet, partitioning, window-function-heavy queries — what Part 9's pruning demo used |
| **dbt-duckdb** | dbt CLI targeting DuckDB as the local warehouse | You want to test *transformation pipelines* (staging → marts, tests, docs) without a real warehouse — this module |
| **Docker Postgres/MySQL** | A real server, containerized | You need something SQLite/DuckDB genuinely can't do: real procedural SQL (Part 11), triggers, concurrent connections, replication behavior |

**The honest ordering:** reach for SQLite/DuckDB first — they cover the overwhelming majority of
what this whole series teaches with zero setup. Reach for Docker only for the one thing they can't
simulate: real server-side procedural execution (Part 11).

## Section 2 — Driving the dbt-duckdb project

This repo's `dbt/` folder is a small, self-contained dbt project: 5 seeds (the same Store dataset),
staging models (light renaming/casting), two mart models (`fct_order_items`,
`customer_order_summary`), one **incremental** model, a macro, and both generic and singular tests.
Run it via `subprocess`, exactly like the source reference project this repo builds on.

In [1]:
import subprocess, os

# Robust to whether the kernel's cwd is the repo root or the notebooks/ folder
# (differs between `jupyter lab` and `jupyter nbconvert --execute`).
for candidate in ("dbt", "../dbt"):
    if os.path.isdir(candidate):
        DBT_DIR = os.path.abspath(candidate)
        break
else:
    raise FileNotFoundError("Could not locate the dbt/ project directory")

env = {**os.environ, "DBT_PROFILES_DIR": DBT_DIR}
print(f"DBT_DIR resolved to: {DBT_DIR}")

result = subprocess.run(
    ["dbt", "build", "--project-dir", DBT_DIR],
    env=env, cwd=DBT_DIR, capture_output=True, text=True,
)
print(result.stdout[-2500:])
if result.returncode != 0:
    print("STDERR:", result.stderr[-2000:])
print(f"\nreturn code: {result.returncode}")
assert result.returncode == 0

DBT_DIR resolved to: /mnt/c/github/sql-for-data-platforms/dbt


emental_orders_order_id ......................... [RUN]
17:51:45  42 of 52 OK created sql table model main.fct_order_items ....................... [OK in 0.29s]
17:51:45  45 of 52 START test assert_line_totals_are_positive ............................ [RUN]
17:51:45  46 of 52 START test not_null_fct_order_items_line_total ........................ [RUN]
17:51:45  43 of 52 PASS not_null_incremental_orders_order_id ............................. [PASS in 0.15s]
17:51:45  44 of 52 PASS unique_incremental_orders_order_id ............................... [PASS in 0.16s]
17:51:45  47 of 52 START test not_null_fct_order_items_order_item_id ..................... [RUN]
17:51:45  48 of 52 START test unique_fct_order_items_order_item_id ....................... [RUN]
17:51:45  45 of 52 PASS assert_line_totals_are_positive .................................. [PASS in 0.17s]
17:51:45  46 of 52 PASS not_null_fct_order_items_line_total .............................. [PASS in 0.18s]
17:51:45  47 of 52 PASS

`dbt build` seeded all 5 tables, built every staging view and mart table (including the new
**incremental** model), and ran every test — generic (`unique`, `not_null`, `accepted_values`,
`relationships`) and singular (`assert_line_totals_are_positive`). All passing means: the data
matches its documented contract, right now, without a human checking it by hand.

In [2]:
# Query the built warehouse directly to confirm the marts materialized correctly
import duckdb
ddb = duckdb.connect(os.path.join(DBT_DIR, "dbt.duckdb"), read_only=True)
ddb.execute("SELECT * FROM customer_order_summary ORDER BY spend_rank LIMIT 5").df()

,customer_id,customer_name,city,membership_level,total_orders,total_spent,avg_order_value,spend_rank
0,12,Liam Johnson,Toronto,vip,4,902.87,225.72,1
1,1,Alice Martin,Toronto,premium,3,659.91,219.97,2
2,7,Grace Kim,Vancouver,vip,3,549.89,183.30,3
3,3,Carol White,Montreal,vip,4,544.89,136.22,4
4,19,Samuel Nguyen,Vancouver,vip,3,539.92,179.97,5


## Section 3 — The incremental model: Part 8's watermark pattern, productionized

`models/marts/incremental_orders.sql` is configured `materialized='incremental'`. First run:
builds the full table from every row in `stg_orders`. Every run after: dbt wraps the model's
`SELECT` in an `is_incremental()` block that filters the source to only rows newer than
`max(order_date)` already in the target — **the exact watermark pattern from Part 8**, expressed as
configuration instead of hand-written procedural logic.

In [3]:
ddb.execute("SELECT COUNT(*) AS row_count, MAX(order_date) AS latest FROM incremental_orders").df()

,row_count,latest
0,40,2024-12-15


In [4]:
ddb.close()

# Run dbt build again -- with no new source rows, the incremental model has nothing to add
result2 = subprocess.run(
    ["dbt", "run", "--select", "incremental_orders", "--project-dir", DBT_DIR],
    env=env, cwd=DBT_DIR, capture_output=True, text=True,
)
print(result2.stdout[-1200:])

17:52:09  Running with dbt=1.12.0
17:52:10  Registered adapter: duckdb=1.10.1
17:52:24  Found 8 models, 39 data tests, 5 seeds, 487 macros
17:52:24  
17:52:24  Concurrency: 4 threads (target='dev')
17:52:24  
17:52:27  1 of 1 START sql incremental model main.incremental_orders ..................... [RUN]
17:52:28  1 of 1 OK created sql incremental model main.incremental_orders ................ [OK in 0.26s]
17:52:28  
17:52:28  Finished running 1 incremental model in 0 hours 0 minutes and 4.05 seconds (4.05s).
17:52:28  
17:52:28  Completed successfully
17:52:28  
17:52:28  Done. PASS=1 WARN=0 ERROR=0 SKIP=0 NO-OP=0 REUSED=0 TOTAL=1



## Section 4 — Tests and docs as lightweight governance

Three things a data platform's governance layer usually provides — access control, a catalog, and
lineage — dbt gives you two of the three, for free, on a laptop:

- **Catalog-lite:** every model's `description` and column tests in `_staging.yml`/`_marts.yml`
  (already in this project) *are* a searchable data dictionary once you run `dbt docs generate`.
- **Lineage-lite:** dbt builds its dependency graph from every `{{ ref(...) }}` call — the same DAG
  concept as Unity Catalog/Purview lineage, just computed from your own model code instead of a
  platform's metadata service.
- **Access control** is the one piece dbt genuinely can't simulate locally — that's inherently a
  platform-level concern (who can query what), not a transformation-layer one.

In [5]:
result3 = subprocess.run(
    ["dbt", "docs", "generate", "--project-dir", DBT_DIR],
    env=env, cwd=DBT_DIR, capture_output=True, text=True,
)
print(result3.stdout[-800:])
print("catalog.json + manifest.json (the lineage graph) now in dbt/target/ -- 'dbt docs serve' renders them as a browsable site")

17:52:46  Running with dbt=1.12.0
17:52:47  Registered adapter: duckdb=1.10.1
17:53:00  Found 8 models, 39 data tests, 5 seeds, 487 macros
17:53:00  
17:53:00  Concurrency: 4 threads (target='dev')
17:53:00  
17:53:05  Building catalog
17:53:05  Catalog written to /mnt/c/github/sql-for-data-platforms/dbt/target/catalog.json

catalog.json + manifest.json (the lineage graph) now in dbt/target/ -- 'dbt docs serve' renders them as a browsable site


## Best Practices — Local Simulation & Governance

- Match the tool to the question: SQLite/DuckDB for query logic, dbt-duckdb for *pipeline* logic,
  Docker only when you need a real server-side feature these can't simulate.
- Write the generic tests (`unique`, `not_null`, `relationships`, `accepted_values`) before the
  first singular test — they cover the majority of real data-quality bugs for a few lines of YAML.
- Treat `dbt docs generate` as a byproduct you get automatically from documenting models well, not
  as separate work — the lineage graph and the catalog are both derived, not hand-maintained.

## Next

**Part 11 — Procedural SQL** is the one module in this series that needs a real server — it reuses
the "Docker Postgres" option from Section 1 above for a genuinely runnable PL/pgSQL demo.